**EEG–Eye-Tracking Multimodal Research | Presented by: Reza Saadatyar (2026)**

**06 | Validation Selection, Robustness and Held-Out Evaluation**

![Validation Selection, Robustness and Held-Out Evaluation](../../images/06_robustness_and_final_evaluation.svg)

**Objective:** reconcile all candidate Validation predictions,
       freeze the primary model using the declared ranking, then conduct
       one held-out person-disjoint Test evaluation. **Inputs:** fixed
       `01_*`–`05_*` cohort, features, models and prediction tables.
       **Scientific boundary:** source task discrimination is not a direct
       estimate of reading comprehension or cognitive state.

       For an initial small-cohort engineering run, leave
       `RUN_FINAL_TEST = False`. Only set it to `True` after the full source
       cohort, modelling specification and primary-selection policy are
       frozen. This preserves the Test during iterative development.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "-q", repo_url, str(repo_path)], check=True
    )

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eeg_eye_tracking_fusion")

# ---------------------------- 2. Standard Library ----------------------------
import importlib.util
import json
import random
import shutil
import subprocess
import sys
from pathlib import Path

# ---------------------- 3. Colab Dependency Preparation ----------------------
if importlib.util.find_spec("mne") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "mne>=1.6,<2"],
        check=True,
    )

# ----------------- 4. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 5. Machine Learning and Deep Learning -------------------
import joblib
from sklearn.metrics import ConfusionMatrixDisplay
import torch

# ------------------- 6. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style

# ----------------- 7. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from models import (
    MODEL_ORDER, CLASSICAL_ORDER, NEURAL_ORDER, CLASS_TO_INDEX,
    metrics, matrix, probabilities, record_predictions,
    torch_network, predict_neural, feature_columns,
)

# ---------------------------- 8. Reproducibility -----------------------------
SEED = 17
random.seed(SEED)
np.random.seed(SEED)

# ------------------- 9. Output Directory and Setup Summary -------------------
work = root / "eeg_eye_tracking_fusion"
output_dir = work / "output"
output_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path("/content/zuco2_source_cache")
raw_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Project outputs:", output_dir)
print("Temporary source cache:", raw_dir)
print("Python:", sys.version.split()[0])
print("Free disk (GiB):", round(
    shutil.disk_usage("/content").free / 2**30, 2,
))


**02. Load Upstream Models and Reconcile Validation Records**

Every candidate must use exactly the same Validation window
          identifiers and original task labels; incompatible/mixed outputs
          are rejected before any primary selection or Test scoring.

In [ ]:
# ========= 02. Load Upstream Models and Reconcile Validation Records =========


table = pd.read_csv(output_dir / '03_features.csv')
assert table.window_id.is_unique
valid = table.loc[table.split.eq('validation')].copy()
test = table.loc[table.split.eq('test')].copy()
y_valid = valid.task.map(CLASS_TO_INDEX).to_numpy()
if min(valid.participant_id.nunique(),
       test.participant_id.nunique()) < 1:
    raise ValueError('Missing Validation or Test participants')
split_people = table.groupby('participant_id').split.nunique()
if not split_people.eq(1).all():
    raise ValueError('Participant overlap across split roles')
source = pd.concat([
    pd.read_csv(output_dir / '04_validation_predictions.csv'),
    pd.read_csv(output_dir / '05_validation_predictions.csv')],
    ignore_index=True)
summaries = pd.concat([
    pd.read_csv(output_dir / '04_validation_metrics.csv'),
    pd.read_csv(output_dir / '05_validation_metrics.csv')],
    ignore_index=True)
if set(source.model) != set(MODEL_ORDER):
    raise ValueError('Expected every classical and neural candidate')
for name in MODEL_ORDER:
    part = source.loc[source.model.eq(name)]
    if (len(part) != len(valid) or part.window_id.duplicated().any()
            or set(part.window_id) != set(valid.window_id)):
        raise ValueError(f'Validation source IDs mismatch: {name}')
    aligned = part.set_index('window_id').loc[valid.window_id]
    if not np.array_equal(aligned.target_index, y_valid):
        raise ValueError(f'Validation true labels differ: {name}')
    proba = aligned[['p_nr', 'p_tsr']].to_numpy()
    if (not np.isfinite(proba).all() or
            not np.allclose(proba.sum(axis=1), 1, atol=1e-5) or
            not np.array_equal(aligned.pred_index,
                               proba.argmax(axis=1))):
        raise ValueError(f'Validation probability mismatch: {name}')
    recorded = summaries.loc[summaries.model.eq(name)]
    if len(recorded) != 1:
        raise ValueError(f'Expected one metrics row for {name}')
    recomputed = metrics(y_valid, aligned.pred_index.to_numpy())
    for metric, score in recomputed.items():
        if not np.isclose(score, recorded.iloc[0][metric], atol=1e-9):
            raise ValueError(f'Saved {metric} differs: {name}')
print('Verified', len(MODEL_ORDER), 'models on', len(valid),
      'matching Validation windows')


**03. Freeze the Validation-Selected Primary Candidate**

Model order is prespecified in code. The ranking uses fixed-label
          Macro F1, then Balanced Accuracy, then fixed candidate priority;
          Test information is excluded from the selection operation.

In [ ]:
# =========== 03. Freeze the Validation-Selected Primary Candidate ============

ranking = summaries[['model', 'macro_f1', 'balanced_accuracy']].copy()
ranking['priority'] = ranking.model.map(MODEL_ORDER.index)
ranking = ranking.sort_values(
    ['macro_f1', 'balanced_accuracy', 'priority'],
    ascending=[False, False, True]).reset_index(drop=True)
selected_name = str(ranking.iloc[0].model)
protocol = json.loads((output_dir / '01_protocol.json').read_text())
RUN_FINAL_TEST = False  # True only for the frozen final evaluation
evaluation_mode = 'final' if RUN_FINAL_TEST else 'development'
decision = pd.DataFrame([{
    'selected_model': selected_name,
    'validation_macro_f1': float(ranking.iloc[0].macro_f1),
    'validation_n_people': valid.participant_id.nunique(),
    'test_used_in_selection': False,
    'run_mode': evaluation_mode,
    'source_people': table.participant_id.nunique(),
    'dataset_version': protocol['dataset_version'],
}])
decision.to_csv(output_dir / '06_model_selection.csv', index=False)
display(ranking)
print('Frozen Validation primary:', selected_name)
print('Test scoring enabled:', RUN_FINAL_TEST)


**04. Score Frozen Models on the Test Only When Authorised**

Set `RUN_FINAL_TEST=True` in the previous cell **only after**
          the full cohort and all model choices are frozen. In development
          mode, Test remains unexamined; its columns are not used for
          ranking or hyperparameter tuning.

In [ ]:
# ========= 04. Score Frozen Models on the Test Only When Authorised ==========

test_predictions, test_scores = [], []
if RUN_FINAL_TEST:
    y_test = test.task.map(CLASS_TO_INDEX).to_numpy()
    scaling = np.load(output_dir / '05_scaling.npz',
                      allow_pickle=False)
    sequences = np.load(output_dir / '03_temporal_sequences.npz',
                        allow_pickle=False)
    if not np.array_equal(sequences['window_id'],
                          table.window_id.astype(str).to_numpy()):
        raise ValueError('Test window/temporal sequence identity changed')
    test_idx = np.where(table.split.eq('test').to_numpy())[0]

    def standardised(key):
        original = sequences[key]
        median = scaling[f'{key}_median']
        mean = scaling[f'{key}_mean']
        std = scaling[f'{key}_std']
        return ((np.where(np.isfinite(original), original, median)
                 - mean) / std).astype('float32')

    eeg_test = standardised('eeg')[test_idx]
    gaze_test = standardised('gaze')[test_idx]
    probas = {}
    for name in CLASSICAL_ORDER:
        if name == 'late_equal_fusion':
            continue
        group = ('eeg' if name.startswith('eeg_') else 'gaze'
                 if name.startswith('gaze_') or name == 'class_prior'
                 else 'fused')
        fitted = joblib.load(output_dir / f'04_{name}.joblib')
        probas[name] = probabilities(fitted, matrix(test, group))
    probas['late_equal_fusion'] = 0.5 * (
        probas['eeg_logistic'] + probas['gaze_logistic'])
    for name in NEURAL_ORDER:
        checkpoint = torch.load(output_dir / f'05_{name}.pt',
                                map_location='cpu',
                                weights_only=True)
        net = torch_network(name, eeg_test.shape[-1],
                            gaze_test.shape[-1],
                            hidden=int(checkpoint['hidden']))
        net.load_state_dict(checkpoint['state_dict'])
        probas[name] = predict_neural(net, eeg_test, gaze_test)
    for name in MODEL_ORDER:
        score = metrics(y_test, probas[name].argmax(axis=1))
        test_scores.append({
            'model': name, 'n_windows': len(test),
            'n_people': test.participant_id.nunique(),
            'selected_primary': name == selected_name, **score})
        test_predictions.append(record_predictions(test, name,
                                                   probas[name]))
    test_metrics = pd.DataFrame(test_scores)
    prediction_table = pd.concat(test_predictions, ignore_index=True)
    test_metrics.to_csv(output_dir / '06_test_metrics.csv',
                        index=False)
    prediction_table.to_csv(output_dir / '06_test_predictions.csv',
                            index=False)
    display(test_metrics.sort_values('macro_f1', ascending=False))
else:
    print('Development mode: no Test scoring performed.')


**05. Audit Person-Level Test Support and Clustered Uncertainty**

Confidence limits resample *people*, not windows. If only one
          or two independent Test people are available, population-level
          confidence intervals are not claimed.

In [ ]:
# ======= 05. Audit Person-Level Test Support and Clustered Uncertainty =======

if RUN_FINAL_TEST:
    groupwise = []
    for name, part in prediction_table.groupby('model'):
        for person, records in part.groupby('participant_id'):
            groupwise.append({'model': name,
                              'participant_id': person,
                              'n_windows': len(records),
                              **metrics(records.target_index,
                                        records.pred_index)})
    grouped = pd.DataFrame(groupwise)
    grouped.to_csv(output_dir / '06_person_metrics.csv', index=False)
    primary = prediction_table.loc[
        prediction_table.model.eq(selected_name)].copy()
    people = primary.participant_id.unique()
    ci = []
    if len(people) >= 3:
        rng = np.random.default_rng(SEED)
        for repeat in range(500):
            sampled = rng.choice(people, len(people), replace=True)
            samples = pd.concat([
                primary.loc[primary.participant_id.eq(p)]
                for p in sampled], ignore_index=True)
            ci.append({'bootstrap_id': repeat,
                       **metrics(samples.target_index,
                                 samples.pred_index)})
    uncertainty = pd.DataFrame(ci, columns=[
        'bootstrap_id', 'accuracy', 'macro_f1',
        'balanced_accuracy'])
    uncertainty.to_csv(output_dir / '06_cluster_bootstrap.csv',
                       index=False)
    if not uncertainty.empty:
        print('Cluster bootstrap Macro F1 95% interval:',
              np.quantile(uncertainty.macro_f1, [0.025, 0.975]))
    else:
        print('Too few independent Test people for bootstrap CI.')
else:
    print('No groupwise Test analysis in development mode.')


**06. Inspect Validation Sensitivity Without Retuning Models**

As a descriptive negative control, evaluate the already-fitted
          early-fusion logistic pipeline after masking EEG or gaze features
          with the Train-fitted imputer values. This is **Validation-only**
          and neither refits nor replaces the selected primary model.

In [ ]:
# ======== 06. Inspect Validation Sensitivity Without Retuning Models =========

model = joblib.load(
    output_dir / '04_early_fusion_logistic.joblib')
columns = feature_columns(valid, 'fused')
original = valid[columns].copy()
observations = []
for change in ('unchanged', 'mask_eeg', 'mask_gaze'):
    x = original.copy()
    if change != 'unchanged':
        prefix = 'eeg_' if change == 'mask_eeg' else 'gaze_'
        x.loc[:, [col for col in columns if col.startswith(prefix)]] = np.nan
    prediction = probabilities(model, x.to_numpy(dtype=float))
    observations.append({
        'condition': change, 'model': 'early_fusion_logistic',
        'partition': 'validation', 'n_windows': len(valid),
        'n_people': valid.participant_id.nunique(),
        **metrics(y_valid, prediction.argmax(axis=1))})
robustness = pd.DataFrame(observations)
robustness.to_csv(output_dir / '06_validation_sensitivity.csv',
                  index=False)
display(robustness)


**07. Create Evaluation Figures and Verify Stage-Owned Outputs**

The primary is fixed using Validation, not whichever Test
          candidate happens to rank highest. Only observed Test results
          are plotted; development mode produces a selection figure and
          sensitivity figure without opening the Test.

In [ ]:
# ======= 07. Create Evaluation Figures and Verify Stage-Owned Outputs ========

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ranking.sort_values('macro_f1').plot.barh(
    x='model', y='macro_f1', ax=axes[0], legend=False)
axes[0].set(xlabel='Macro F1', title='Validation model selection',
            xlim=(0, 1.03), ylabel='Candidate')
axes[1].bar(robustness.condition, robustness.macro_f1)
axes[1].set(xlabel='Fixed-feature perturbation',
            ylabel='Validation Macro F1', ylim=(0, 1.05),
            title='Early-fusion sensitivity (not attribution)')
axes[1].tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(output_dir / '06_validation_evidence.png', dpi=170)
plt.show()
if RUN_FINAL_TEST:
    primary = prediction_table.loc[
        prediction_table.model.eq(selected_name)]
    fig, ax = plt.subplots(figsize=(4.6, 4.3))
    ConfusionMatrixDisplay.from_predictions(
        primary.target_index, primary.pred_index,
        labels=[0, 1], display_labels=['NR', 'TSR'],
        ax=ax, colorbar=False)
    ax.set_title('Primary held-out Test confusion')
    fig.tight_layout()
    fig.savefig(output_dir / '06_primary_test_confusion.png',
                dpi=170)
    plt.show()
    assert prediction_table.groupby('model').window_id.nunique().eq(
        len(test)).all()
    assert set(test_metrics.model) == set(MODEL_ORDER)
    print('Saved main Test evidence for',
          test.participant_id.nunique(), 'independent people.')
else:
    print('Final Test outputs are deferred until RUN_FINAL_TEST=True.')


**08. Final Interpretation and Reproducibility**

Report uncertainty at participant level, stimulus and task
          confounding, metadata-derived alignment limits, and the small
          independent sample. Fusion gain is a hypothesis to be tested;
          no numerical result is asserted before a genuine notebook run.

In [ ]:
# =============== 08. Final Interpretation and Reproducibility ================

assert (output_dir / '06_model_selection.csv').exists()
assert (output_dir / '06_validation_sensitivity.csv').exists()
if RUN_FINAL_TEST:
    assert (output_dir / '06_test_metrics.csv').exists()
    print('Stage 06 final evaluation files are ready for review.')
else:
    print('Development evidence only. Keep final Test untouched.')
print('Scientific report:',
      work / 'RESEARCH_METHODS_AND_EVIDENCE.md')

# ---------------------- 2. Package Stage-Owned Outputs -----------------------
download_outputs_from_colab(
    "eeg_eye_tracking_fusion/output", "06"
)
